# Laptop ABSA — data exploration

Walks the corpus the training package loads: class balance, aspect
distribution, multi-aspect sentences, mixed-polarity sentences, and what
the encoder actually feeds the model.

Run from `training/notebooks/`.

In [1]:
import sys
from pathlib import Path

TRAINING_ROOT = Path.cwd().parent
sys.path.insert(0, str(TRAINING_ROOT))

import pandas as pd

from src.data.dataset import (
    LabelMap, label_distribution, load_official_test, load_raw_csv, split_train_val,
)

pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 80)
label_map = LabelMap()
label_map.label2id

{'negative': 0, 'neutral': 1, 'positive': 2}

## Load

`load_raw_csv` drops `conflict`, realigns the 7 column-shifted rows and
validates `text[from:to] == aspect` on every row. Watch the log line.

In [2]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s', force=True)

RAW = TRAINING_ROOT / 'data' / 'raw'
train_full = load_raw_csv(RAW / 'Laptop_Train_v2.csv')
test = load_official_test(RAW / 'Laptops_Test_Gold.csv')

print(f'train {len(train_full)} rows, test {len(test)} rows')
train_full.head()

WARNING | Laptop_Train_v2.csv: realigned 7 column-shifted row(s)


INFO | Laptop_Train_v2.csv: 2358 rows -> 2313 usable (dropped 45 conflict), 1456 unique sentences


INFO | Laptops_Test_Gold.csv: 654 rows -> 638 usable (dropped 16 conflict), 411 unique sentences


train 2313 rows, test 638 rows


,review_id,text,aspect,label,from,to
0,2339,I charge it at night and skip taking the cord with me because of the good ba...,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord with me because of the good ba...,battery life,positive,74,86
2,1316,The tech guy then said the service center does not do 1-to-1 exchange and I ...,service center,negative,27,41
3,1316,The tech guy then said the service center does not do 1-to-1 exchange and I ...,"""sales"" team",negative,109,121
4,1316,The tech guy then said the service center does not do 1-to-1 exchange and I ...,tech guy,neutral,4,12


## Class balance

`positive` dominates and `neutral` is the minority — which is why the
trainer weights the loss by inverse frequency and selects on macro F1.

In [3]:
balance = pd.DataFrame({
    'train': label_distribution(train_full),
    'test': label_distribution(test),
})
balance['train %'] = (100 * balance['train'] / balance['train'].sum()).round(1)
balance['test %'] = (100 * balance['test'] / balance['test'].sum()).round(1)
balance

,train,test,train %,test %
negative,866,128,37.4,20.1
neutral,460,169,19.9,26.5
positive,987,341,42.7,53.4


## Aspects per sentence

The reason the split is grouped by sentence: many sentences carry several
aspects, and some carry *opposing* sentiments across them.

In [4]:
per_sentence = train_full.groupby('text').size()
print(f'{train_full["text"].nunique()} unique sentences for {len(train_full)} rows')
print(f'mean {per_sentence.mean():.2f} aspects/sentence, max {per_sentence.max()}')
per_sentence.value_counts().sort_index().rename('sentences').to_frame().T

1456 unique sentences for 2313 rows
mean 1.59 aspects/sentence, max 13


,1,2,3,4,5,6,7,8,13
sentences,913,342,134,45,10,7,3,1,1


In [5]:
# Sentences whose aspects disagree -- the examples a sentence-level model cannot get right.
mixed = train_full.groupby('text')['label'].nunique()
mixed_texts = mixed[mixed > 1].index
print(f'{len(mixed_texts)} sentences carry more than one polarity '
      f'({100 * len(mixed_texts) / train_full["text"].nunique():.1f}% of sentences)')

train_full[train_full['text'].isin(mixed_texts[:3])][['text', 'aspect', 'label']]

164 sentences carry more than one polarity (11.3% of sentences)


,text,aspect,label
789,(I had been a Windows/Linux user before this) I love the size because the sc...,size,positive
790,(I had been a Windows/Linux user before this) I love the size because the sc...,screen,positive
791,(I had been a Windows/Linux user before this) I love the size because the sc...,Windows,neutral
792,(I had been a Windows/Linux user before this) I love the size because the sc...,Linux,neutral
807,"(If you ever see the spinning beachball come up when you think it shouldn't,...",disk throughput,neutral
808,"(If you ever see the spinning beachball come up when you think it shouldn't,...",spinning beachball,negative
2164,", Applications respond immediately (not like the tired MS applications).",Applications,positive
2165,", Applications respond immediately (not like the tired MS applications).",MS applications,negative


## Most frequent aspects

In [6]:
train_full['aspect'].str.lower().value_counts().head(20).rename('count').to_frame()

,count
aspect,
screen,60
price,56
use,53
battery life,52
keyboard,50
battery,47
programs,37
features,35
software,33


In [7]:
# Polarity profile of the most common aspects.
top = train_full['aspect'].str.lower().value_counts().head(10).index
profile = (train_full.assign(aspect_lc=train_full['aspect'].str.lower())
           .query('aspect_lc in @top')
           .pivot_table(index='aspect_lc', columns='label', aggfunc='size', fill_value=0))
profile.loc[top]

label,negative,neutral,positive
aspect,,,
screen,23,6,31
price,8,2,46
use,3,1,49
battery life,17,3,32
keyboard,19,8,23
battery,32,5,10
programs,9,9,19
features,5,8,22
software,17,7,9


## Sentence and aspect length

In [8]:
lengths = pd.DataFrame({
    'sentence_chars': train_full['text'].str.len(),
    'sentence_words': train_full['text'].str.split().str.len(),
    'aspect_words': train_full['aspect'].str.split().str.len(),
})
lengths.describe(percentiles=[0.5, 0.9, 0.99]).round(1)

,sentence_chars,sentence_words,aspect_words
count,2313.0,2313.0,2313.0
mean,104.8,19.3,1.5
std,57.1,10.6,0.7
min,11.0,2.0,1.0
50%,95.0,17.0,1.0
90%,179.0,33.0,2.0
99%,297.0,58.0,4.0
max,464.0,78.0,6.0


## The split

Grouped by sentence, so no sentence appears on both sides.

In [9]:
train_df, val_df = split_train_val(train_full, val_fraction=0.1, seed=42)

leak = set(train_df['text']) & set(val_df['text'])
print(f'overlapping sentences: {len(leak)}')
print(f'val is {100 * len(val_df) / len(train_full):.1f}% of rows')

pd.DataFrame({'train': label_distribution(train_df), 'val': label_distribution(val_df)})

INFO | split: 2081 train / 232 val rows (1313 / 143 sentences)


overlapping sentences: 0
val is 10.0% of rows


,train,val
negative,784,82
neutral,421,39
positive,876,111


## What the model actually sees

The encoder builds `[CLS] text [SEP] aspect [SEP]` plus two disjoint masks.
Needs `bert-base-uncased` available locally or via the hub.

In [10]:
from src.preprocessing.tokenizer import ABSAEncoder

encoder = ABSAEncoder('bert-base-uncased', max_length=32)
example = encoder.encode('The battery life is amazing but the screen is dim', 'battery life', label=2)

{k: tuple(v.shape) for k, v in example.items()}

INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"


INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"


INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"


INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"


INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


{'input_ids': (32,),
 'attention_mask': (32,),
 'token_type_ids': (32,),
 'context_mask': (32,),
 'aspect_mask': (32,),
 'labels': ()}

In [11]:
tokens = encoder.tokenizer.convert_ids_to_tokens(example['input_ids'])

pd.DataFrame({
    'token': tokens,
    'type': example['token_type_ids'].tolist(),
    'attn': example['attention_mask'].tolist(),
    'context': example['context_mask'].tolist(),
    'aspect': example['aspect_mask'].tolist(),
}).head(18)

,token,type,attn,context,aspect
0,[CLS],0,1,False,False
1,the,0,1,True,False
2,battery,0,1,True,False
3,life,0,1,True,False
4,is,0,1,True,False
5,amazing,0,1,True,False
6,but,0,1,True,False
7,the,0,1,True,False
8,screen,0,1,True,False
9,is,0,1,True,False


In [12]:
# The two masks never overlap and never select a special or padding token.
ctx, asp = example['context_mask'], example['aspect_mask']
print('overlap          :', bool((ctx & asp).any()))
print('context tokens   :', encoder.tokenizer.convert_ids_to_tokens(example['input_ids'][ctx]))
print('aspect tokens    :', encoder.tokenizer.convert_ids_to_tokens(example['input_ids'][asp]))

overlap          : False
context tokens   : ['the', 'battery', 'life', 'is', 'amazing', 'but', 'the', 'screen', 'is', 'dim']
aspect tokens    : ['battery', 'life']


In [13]:
# One batch exactly as the DataLoader builds it: per-example encoding
# through ABSADataset, then collate_batch stacking it.
#
# The weights below are inverse-frequency over *these 64 val rows*, shown to
# demonstrate the call -- not the figures training uses, which come from the
# full 2081-row training split.
from src.data.dataset import ABSADataset, collate_batch

dataset = ABSADataset(val_df.head(64), encoder, label_map)
batch = collate_batch([dataset[i] for i in range(8)])

print(dataset)
print({k: tuple(v.shape) for k, v in batch.items()})
print('class weights:', dict(zip(label_map.labels, dataset.class_weights().tolist())))

ABSADataset(n=64, labelled=True, labels=('negative', 'neutral', 'positive'))
{'input_ids': (8, 32), 'attention_mask': (8, 32), 'token_type_ids': (8, 32), 'context_mask': (8, 32), 'aspect_mask': (8, 32), 'labels': (8,)}
class weights: {'negative': 1.0991276502609253, 'neutral': 1.3346550464630127, 'positive': 0.5662172436714172}


## The attention, on an untrained model

`BertAspectAttention` scores every context token against the aspect
vector. Untrained the weights are near-uniform, but the *shape* of the
output is what matters here: one weight per context token, exactly zero
outside the mask, summing to 1.

In [14]:
import torch
from transformers import BertConfig, BertModel

from src.models.absa_model import BertAspectAttention

# A tiny encoder keeps this cell instant and offline.
tiny = BertModel(BertConfig(vocab_size=encoder.tokenizer.vocab_size, hidden_size=32,
                            num_hidden_layers=2, num_attention_heads=2,
                            intermediate_size=37, max_position_embeddings=64))
model = BertAspectAttention(encoder=tiny, num_labels=3)

out = model(**{k: v.unsqueeze(0) for k, v in example.items() if k != 'labels'})
print('logits   :', tuple(out['logits'].shape))
print('attention:', tuple(out['attention'].shape))
w = out['attention'][0].detach()
print('sums to 1:', torch.allclose(w.sum(), torch.tensor(1.0), atol=1e-5))
print('zero off-mask:', float(w[~example['context_mask']].abs().max()))

logits   : (1, 3)
attention: (1, 32)
sums to 1: True
zero off-mask: 0.0


In [15]:
# Weight per context token -- after training these concentrate on the
# sentiment-bearing words for this particular aspect.
ctx = example['context_mask']
pd.DataFrame({
    'token': [t for t, k in zip(tokens, ctx.tolist()) if k],
    'weight': [round(x, 4) for x, k in zip(w.tolist(), ctx.tolist()) if k],
})

,token,weight
0,the,0.1127
1,battery,0.1191
2,life,0.0628
3,is,0.1201
4,amazing,0.0915
5,but,0.0935
6,the,0.1192
7,screen,0.0962
8,is,0.0991
9,dim,0.0859


## Next

```bash
cd ..
python train.py prepare    # split the corpus
python train.py train      # fine-tune -> artifacts/absa-v1/
python train.py evaluate   # score the gold test set
```

Then load the artifact and inspect what the attention picked up:

```python
from src.artifact import load_artifact, predict_pairs

model, enc, lm, cfg = load_artifact('../artifacts/absa-v1')
predict_pairs(model, enc, lm, [
    ('Battery life is amazing but the screen is dim.', 'screen'),
])
```
